# Week 5 - Day 3: Data Cleaning
**Goal:** Clean messy real-world data before analysis.

Real financial data is NEVER clean. Prices come as `"$1,234"` strings,
sectors have inconsistent casing, rows are duplicated, and values are missing.
Today you learn to fix all of that.

In [1]:
import pandas as pd

df = pd.read_csv("data/sgx_messy.csv")
df

,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
0,D05,DBS Group,Banking,"$108,500",41.20,5.1,10.2
1,O39,OCBC Bank,Banking,"$65,200",15.80,5.4,9.8
2,U11,UOB,Banking,"$55,800",34.50,4.8,10.5
3,Z74,Singtel,telecom,"$42,100",3.25,4.9,18.3
4,C6L,Singapore Airlines,Aviation,"$18,900",6.85,NaN,15.7
5,9CI,CapitaLand Investment,Real Estate,"$17,800",3.12,3.8,22.4
6,SE,Sea Limited,Technology,"$35,600",125.40,0.0,45.2
7,GRAB,Grab Holdings,technology,"$14,200",4.85,0.0,NaN
8,A17U,CapitaLand Ascendas REIT,REIT - Industrial,"$12,400",2.78,5.6,18.1
9,J69U,Frasers Logistics Trust,REIT - Industrial,"$4,800",1.22,6.1,14.5


---
## Part 1: Spot the Problems

Before cleaning, you need to find what's wrong.

New concepts:
- `df.shape` — how many rows/columns
- `df.dtypes` — check if types are correct (market_cap should be number, not string)
- `df.isnull().sum()` — count missing values per column
- `df.duplicated().sum()` — count duplicate rows

In [ ]:
# TODO: Check the shape — we expect 20 stocks, but there might be extras
#   print(f"Shape: {df.shape}")
#   print(f"Expected 20 rows, got {df.shape[0]}")

In [2]:
df.shape

(22, 7)

In [3]:
df.shape[0]

22

In [ ]:
# TODO: Check data types — is market_cap_m a number or string?
#   df.dtypes

In [4]:
df.dtypes

ticker             object
name               object
sector             object
market_cap_m       object
price             float64
dividend_yield    float64
pe_ratio          float64
dtype: object

In [ ]:
# TODO: Count missing values per column
#   df.isnull().sum()

In [5]:
df.isnull().sum()

ticker            0
name              0
sector            0
market_cap_m      0
price             1
dividend_yield    1
pe_ratio          1
dtype: int64

In [ ]:
# TODO: Count duplicate rows
#   print(f"Duplicate rows: {df.duplicated().sum()}")

In [6]:
df.duplicated().sum()

np.int64(2)

In [ ]:
# TODO: Check unique sectors — are there casing issues?
#   df["sector"].unique()

In [9]:
df["sector"].unique()

array(['Banking', 'telecom', 'Aviation', 'Real Estate', 'Technology',
       'technology', 'REIT - Industrial', 'REIT - Commercial',
       'Conglomerate', 'agriculture', 'Industrial', 'real estate',
       'REIT - Logistics'], dtype=object)

### Problems found:
You should see 5 issues:
1. **22 rows** instead of 20 (duplicates)
2. **market_cap_m** is `object` (string) not `int64` — because of `"$108,500"` format
3. **Missing values** — dividend_yield (1 missing), pe_ratio (1 missing), price (1 missing)
4. **Inconsistent sector casing** — `"telecom"` vs `"Telecom"`, `"real estate"` vs `"Real Estate"`
5. **Duplicate rows** — DBS and OCBC appear twice

Let's fix them one by one.

---
## Part 2: Remove Duplicates

New concepts:
- `df.duplicated()` — True/False for each row (True = it's a duplicate of an earlier row)
- `df.drop_duplicates()` — remove duplicate rows, keep the first occurrence
- Always reassign: `df = df.drop_duplicates()` (or use `inplace=True`)

In [ ]:
# TODO: Show which rows are duplicates
#   df[df.duplicated()]

In [10]:
df[df.duplicated()]

,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
13,D05,DBS Group,Banking,"$108,500",41.2,5.1,10.2
21,O39,OCBC Bank,Banking,"$65,200",15.8,5.4,9.8


In [ ]:
# TODO: Remove duplicates and verify
#   df = df.drop_duplicates()
#   print(f"After removing duplicates: {df.shape[0]} rows")

In [12]:
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape[0]} rows")

After removing duplicates: 20 rows


---
## Part 3: Fix Inconsistent Strings

New concepts:
- `df["col"].str.title()` — capitalize first letter of each word ("real estate" → "Real Estate")
- `df["col"].str.lower()` — all lowercase
- `df["col"].str.upper()` — all uppercase
- `df["col"].str.strip()` — remove leading/trailing whitespace

In [ ]:
# TODO: Check the sector values before fixing
#   df["sector"].value_counts()

In [13]:
df["sector"].value_counts()

sector
Banking              3
REIT - Industrial    3
REIT - Commercial    2
Conglomerate         2
Industrial           2
telecom              1
Aviation             1
Real Estate          1
Technology           1
technology           1
agriculture          1
real estate          1
REIT - Logistics     1
Name: count, dtype: int64

In [14]:
# TODO: Fix sector casing — use .str.title() to standardize
#   df["sector"] = df["sector"].str.title()
#   df["sector"].value_counts()
df["sector"] = df["sector"].str.title()
df["sector"].value_counts()

sector
Banking              3
Reit - Industrial    3
Real Estate          2
Technology           2
Reit - Commercial    2
Conglomerate         2
Industrial           2
Telecom              1
Aviation             1
Agriculture          1
Reit - Logistics     1
Name: count, dtype: int64

**Note:** `.str.title()` works for most cases, but it turns
`"REIT - Industrial"` into `"Reit - Industrial"`. We need to fix that.

In [ ]:
# TODO: Fix REIT casing — replace "Reit" back to "REIT"
#   df["sector"] = df["sector"].str.replace("Reit", "REIT")
#   df["sector"].value_counts()

In [15]:
df["sector"] = df["sector"].str.replace("Reit", "REIT")
df["sector"].value_counts()

sector
Banking              3
REIT - Industrial    3
Real Estate          2
Technology           2
REIT - Commercial    2
Conglomerate         2
Industrial           2
Telecom              1
Aviation             1
Agriculture          1
REIT - Logistics     1
Name: count, dtype: int64

---
## Part 4: Clean market_cap_m — String to Number

New concepts:
- `df["col"].str.replace("$", "")` — remove dollar sign
- `df["col"].str.replace(",", "")` — remove commas
- `df["col"].astype(int)` — convert string to integer
- Chain them: `.str.replace(...).str.replace(...).astype(int)`

In [ ]:
# TODO: Look at current market_cap_m values — they're strings like "$108,500"
#   print(df["market_cap_m"].dtype)
#   df["market_cap_m"].head()

In [20]:
print(df["market_cap_m"].dtype)
df["market_cap_m"].head()

object


0    $108,500
1     $65,200
2     $55,800
3     $42,100
4     $18,900
Name: market_cap_m, dtype: object

In [ ]:
# TODO: Clean market_cap_m — remove $ and commas, convert to int
#   df["market_cap_m"] = df["market_cap_m"].str.replace("$", "").str.replace(",", "").astype(int)
#   print(df["market_cap_m"].dtype)
#   df["market_cap_m"].head()

In [21]:
df["market_cap_m"] = df["market_cap_m"].str.replace("$", "").str.replace(",", "").astype(int)
print(df["market_cap_m"].dtype)
df["market_cap_m"].head()

int64


0    108500
1     65200
2     55800
3     42100
4     18900
Name: market_cap_m, dtype: int64

---
## Part 5: Handle Missing Values

New concepts:
- `df.isnull().sum()` — count missing values per column
- `df[df["col"].isnull()]` — show rows with missing values
- `df["col"].fillna(value)` — replace missing with a specific value
- `df["col"].fillna(df["col"].median())` — replace with median (good for numbers)
- `df.dropna()` — remove rows with ANY missing value (use carefully!)

In [ ]:
# TODO: Check which rows have missing values
#   print("Missing values per column:")
#   print(df.isnull().sum())
#   print()
#   print("Rows with missing data:")
#   df[df.isnull().any(axis=1)]

In [22]:
df.isnull().sum()

ticker            0
name              0
sector            0
market_cap_m      0
price             1
dividend_yield    1
pe_ratio          1
dtype: int64

In [23]:
df[df.isnull().any(axis=1)]

,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
4,C6L,Singapore Airlines,Aviation,18900,6.85,NaN,15.7
7,GRAB,Grab Holdings,Technology,14200,4.85,0.0,NaN
14,J36,Jardine Matheson,Conglomerate,28500,NaN,3.5,12.1


In [ ]:
# TODO: Fill missing price with median price
#   median_price = df["price"].median()
#   print(f"Median price: {median_price}")
#   df["price"] = df["price"].fillna(median_price)

In [24]:
median_price = df["price"].median()

In [28]:
median_price

3.42

In [26]:
df["price"] = df["price"].fillna(median_price)

In [ ]:
# TODO: Fill missing dividend_yield with 0 (assume no dividend if not reported)
#   df["dividend_yield"] = df["dividend_yield"].fillna(0)

In [30]:
df["dividend_yield"] = df["dividend_yield"].fillna(0)

In [ ]:
# TODO: Fill missing pe_ratio with median PE
#   median_pe = df["pe_ratio"].median()
#   print(f"Median PE: {median_pe}")
#   df["pe_ratio"] = df["pe_ratio"].fillna(median_pe)

In [31]:
median_pe = df["pe_ratio"].median()
df["pe_ratio"] = df["pe_ratio"].fillna(median_pe)

In [34]:
median_pe

14.2

In [32]:
df

,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
0,D05,DBS Group,Banking,108500,41.20,5.1,10.2
1,O39,OCBC Bank,Banking,65200,15.80,5.4,9.8
2,U11,UOB,Banking,55800,34.50,4.8,10.5
3,Z74,Singtel,Telecom,42100,3.25,4.9,18.3
4,C6L,Singapore Airlines,Aviation,18900,6.85,0.0,15.7
5,9CI,CapitaLand Investment,Real Estate,17800,3.12,3.8,22.4
6,SE,Sea Limited,Technology,35600,125.40,0.0,45.2
7,GRAB,Grab Holdings,Technology,14200,4.85,0.0,14.2
8,A17U,CapitaLand Ascendas REIT,REIT - Industrial,12400,2.78,5.6,18.1
9,J69U,Frasers Logistics Trust,REIT - Industrial,4800,1.22,6.1,14.5


In [ ]:
# TODO: Verify no more missing values
#   df.isnull().sum()

In [33]:
df.isnull().sum()

ticker            0
name              0
sector            0
market_cap_m      0
price             0
dividend_yield    0
pe_ratio          0
dtype: int64

---
## Part 6: Save Cleaned Data & Verify

New concepts:
- `df.to_csv("file.csv", index=False)` — save DataFrame to CSV
- `index=False` — don't save the row numbers as a column

In [ ]:
# TODO: Final check — print a summary of the cleaned data
#   print(f"Rows: {df.shape[0]} (expected 20)")
#   print(f"Missing values: {df.isnull().sum().sum()}")
#   print(f"Duplicates: {df.duplicated().sum()}")
#   print(f"market_cap_m type: {df['market_cap_m'].dtype}")
#   print(f"Sectors: {sorted(df['sector'].unique())}")
#   print()
#   df.head()

In [35]:
df.shape[0]

20

In [39]:
sorted(df['sector'].unique())

['Agriculture',
 'Aviation',
 'Banking',
 'Conglomerate',
 'Industrial',
 'REIT - Commercial',
 'REIT - Industrial',
 'REIT - Logistics',
 'Real Estate',
 'Technology',
 'Telecom']

In [ ]:
# TODO: Save the cleaned data
#   df.to_csv("data/sgx_cleaned.csv", index=False)
#   print("Saved to data/sgx_cleaned.csv")

In [40]:
df.to_csv("data/sgx_cleaned.csv", index=False)
print("Saved to data/sgx_cleaned.csv")

Saved to data/sgx_cleaned.csv


In [ ]:
# TODO: Verify — reload and check it's clean
#   clean = pd.read_csv("data/sgx_cleaned.csv")
#   print(f"Reloaded: {clean.shape}")
#   print(f"Types:\n{clean.dtypes}")
#   clean.head()

In [41]:
clean = pd.read_csv("data/sgx_cleaned.csv")
print(f"Reloaded: {clean.shape}")
print(f"Types:\n{clean.dtypes}")
clean.head()

Reloaded: (20, 7)
Types:
ticker             object
name               object
sector             object
market_cap_m        int64
price             float64
dividend_yield    float64
pe_ratio          float64
dtype: object


,ticker,name,sector,market_cap_m,price,dividend_yield,pe_ratio
0,D05,DBS Group,Banking,108500,41.20,5.1,10.2
1,O39,OCBC Bank,Banking,65200,15.80,5.4,9.8
2,U11,UOB,Banking,55800,34.50,4.8,10.5
3,Z74,Singtel,Telecom,42100,3.25,4.9,18.3
4,C6L,Singapore Airlines,Aviation,18900,6.85,0.0,15.7
